In [0]:
%sql
CREATE VOLUME IF NOT EXISTS football.bronze.chk

In [0]:
%python
from pyspark.sql import functions as F

def run_bronze():
    return (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "text")
        .load("/Volumes/football/bronze/landing/")
        .select(F.col("value").alias("payload"),
                F.col("_metadata.file_path").alias("source_file"),
                F.current_timestamp().alias("bronze_ts"))
        .writeStream
        .option("checkpointLocation", "/Volumes/football/bronze/chk/events")
        .trigger(availableNow=True)          # required on serverless
        .toTable("football.bronze.events_raw"))

In [0]:
%python
q = run_bronze()
q.awaitTermination()
print("Bronze done. Rows processed:", q.lastProgress["numInputRows"] if q.lastProgress else 0)